In [1]:
from pypdf import PdfReader
from elasticsearch import Elasticsearch
import re

In [2]:
es = Elasticsearch("http://localhost:9200")

In [3]:
index_name = "python_book"
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
es.indices.create(index=index_name)


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'python_book'})

In [4]:
pdf_path = "book2.pdf" 

reader = PdfReader(pdf_path)
print(f"Pages:  {len(reader.pages)}")

Pages:  9


In [5]:
doc_id = 1
for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    if text:
        paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 30]
        for para in paragraphs:
            es.index(index=index_name, id=doc_id, document={
                "page": i,
                "paragraph": para
            })
            print(f"[{doc_id}] Indexed paragraph  (page {i})")
            doc_id += 1
    else:
        print(f"⚠️ Page {i} doesn't have text")

print(f"\n✅ Complete indexing. Total documents: {doc_id - 1}")

[1] Indexed paragraph  (page 1)
[2] Indexed paragraph  (page 2)
[3] Indexed paragraph  (page 3)
[4] Indexed paragraph  (page 4)
[5] Indexed paragraph  (page 5)
[6] Indexed paragraph  (page 6)
[7] Indexed paragraph  (page 7)
[8] Indexed paragraph  (page 8)
[9] Indexed paragraph  (page 9)

✅ Complete indexing. Total documents: 9


In [6]:
#First 5
response = es.search(index=index_name, size=5, query={"match_all": {}})

print(f"{response['hits']['total']['value']} documents found:\n")

for hit in response["hits"]["hits"]:
    doc = hit["_source"]
    print(f"ID: {hit['_id']}")
    print(f"Page: {doc['page']}")
    print(f"Content:\n{doc['paragraph'][:300]}...")
    print("-" * 50)

9 documents found:

ID: 1
Page: 1
Content:
3D U-Net: Learning Dense Volumetric
Segmentation from Sparse Annotation
¨Ozg¨un C¸i¸cek1,2(B), Ahmed Abdulkadir1,4,S o e r e nS .L i e n k a m p2,3,
Thomas Brox1,2, and Olaf Ronneberger1,2,5
1 Computer Science Department, University of Freiburg, Freiburg, Germany
cicek@cs.uni-freiburg.de
2 BIOSS Cen...
--------------------------------------------------
ID: 2
Page: 2
Content:
Volumetric Segmentation with the 3D U-Net 425
raw image manual sparse annotation dense segmentation
train and 
apply 
3D u-net
raw image
apply trained 3D u-net
dense segmentation
a
b
Fig. 1. Application scenarios for volumetric segmentation with the 3D u-net. (a) Semi-
automated segmentation: the us...
--------------------------------------------------
ID: 3
Page: 3
Content:
426 ¨O. C¸i¸cek et al.
We show the successful application of the proposed method on diﬃcult con-
focal microscopic data set of the Xenopus kidney. During its development, the
Xenopus kidney forms a com

## Search using ElasticSearch

In [7]:
keyword = "microscopic"  

query = {
    "query": {
        "match": {
            "paragraph": keyword
        }
    }
}

response = es.search(index=index_name, body=query, size=10)

print(f"🔍 Resultados para la palabra clave '{keyword}':")
print(f"Se encontraron {response['hits']['total']['value']} documentos:\n")

for hit in response["hits"]["hits"]:
    doc = hit["_source"]
    print(f"ID: {hit['_id']}")
    print(f"Página: {doc['page']}")
    print(f"Párrafo:\n{doc['paragraph'][:300]}...")
    print("-" * 60)

🔍 Resultados para la palabra clave 'microscopic':
Se encontraron 1 documentos:

ID: 3
Página: 3
Párrafo:
426 ¨O. C¸i¸cek et al.
We show the successful application of the proposed method on diﬃcult con-
focal microscopic data set of the Xenopus kidney. During its development, the
Xenopus kidney forms a complex structure [7] which limits the applicability of
pre-deﬁned parametric models. First we provide...
------------------------------------------------------------
